# 04 - Demo content for R1043

**Owner:** Daria  |  **Output:** `data/synthetic/demo_R1043_panel.json` for Diana's screens

Creates a believable synthetic panel for the demo risk: probability estimates, trigger agreement and a few gaps experts mention. All answers are made up. The risk itself, its rating and its triggers are real.

## 0. Setup

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# Find the repo folder (the one that contains src/risk_tools.py), wherever this notebook is run from
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "risk_tools.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError(f"Could not find src/risk_tools.py above {Path.cwd()}. Put risk_tools.py in the src/ folder of the repo.")
sys.path.append(str(ROOT / "src"))
from risk_tools import *
print("Repo folder:", ROOT)


Repo folder: c:\Users\iliev\Documents\github\HFG8-ACAPS-Ground-Truth


In [2]:
PROCESSED = ROOT / "data" / "processed"
OUT_DIR   = ROOT / "data" / "synthetic"
RISK_ID   = "R1043"
PANEL     = 7
SEED      = 3

## 1. The real risk and its triggers

In [3]:
risks = pd.read_csv(PROCESSED / "risks_clean.csv")
trig = pd.read_csv(PROCESSED / "triggers.csv")
risk = risks[risks["risk_id"] == RISK_ID]
assert not risk.empty, f"{RISK_ID} not found - run notebook 02 first"
risk = risk.iloc[0]
triggers = trig[trig["risk_id"] == RISK_ID].to_dict("records")
print(risk["risk_title"])
print(risk["probability"], "->", risk["acaps_p"], "|", len(triggers), "triggers")
pd.DataFrame(triggers)[["trigger_id", "title"]]

Escalating cross-border insecurity linked to the Sudan conflict increases protection risks and humanitarian needs in eastern Chad
High -> 0.7 | 3 triggers


,trigger_id,title
0,R1043 - T0618,Expansion of cross-border attacks and insecurity
1,R1043 - T0619,Attacks on strategic infrastructure in Chad
2,R1043 - T0620,Renewed proxy mobilisation and cross-border armed group activity


## 2. Synthetic panel
Estimates cluster around ACAPS's rating with some spread. Trigger agreement is set by hand so the demo shows one clear agreement, one partial and one split.

In [4]:
rng = np.random.default_rng(SEED)
experts = simulate_experts(PANEL, seed=SEED)[["expert_id", "pseudonym", "anonymity_level"]]

base = float(risk["acaps_p"]) if pd.notna(risk["acaps_p"]) else 0.7
estimates = np.clip(np.round(rng.normal(base + 0.02, 0.09, PANEL) * 20) / 20, 0.05, 0.95)

target_agree = [6, 5, 3]            # how many of the panel agree with trigger 1, 2, 3 ...
votes = {}
for i, t in enumerate(triggers):
    k = target_agree[i % len(target_agree)]
    agree_ids = set(rng.permutation(experts["expert_id"])[:k])
    votes[t["trigger_id"]] = {e: (e in agree_ids) for e in experts["expert_id"]}

gaps = [
    {"text": "Seasonal flooding along the border could block aid routes.", "mentioned_by": 2},
    {"text": "Refugee arrivals already exceed what local services can absorb.", "mentioned_by": 1},
    {"text": "Watch for tension between host communities and new arrivals.", "mentioned_by": 1},
]
print("Estimates:", estimates.tolist())

Estimates: [0.9, 0.5, 0.75, 0.65, 0.7, 0.7, 0.55]


## 3. Build and save the JSON

In [5]:
answers = []
for i, e in experts.iterrows():
    answers.append({
        "expert_id": e["expert_id"], "pseudonym": e["pseudonym"], "anonymity_level": e["anonymity_level"],
        "probability_6m": float(estimates[i]),
        "trigger_agree": {tid: bool(v[e["expert_id"]]) for tid, v in votes.items()},
    })

panel_view = {
    "synthetic": True,
    "risk": {"risk_id": RISK_ID, "title": risk["risk_title"], "country": risk["country"],
             "acaps_rating": risk["risk_level"], "acaps_probability": risk["probability"]},
    "panel_size": PANEL,
    "probability": {"median": float(np.median(estimates)), "min": float(estimates.min()),
                    "max": float(estimates.max())},
    "triggers": [{"trigger_id": t["trigger_id"], "title": t["title"],
                  "agree": int(sum(votes[t["trigger_id"]].values())), "of": PANEL} for t in triggers],
    "gaps": gaps,
    "moderator_summary_draft": "Experts agree on the main trigger. Two raise a gap. Review before sharing with the panel.",
    "answers": answers,
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "demo_R1043_panel.json", "w", encoding="utf-8") as f:
    json.dump(panel_view, f, ensure_ascii=False, indent=2)
print("Saved", OUT_DIR / "demo_R1043_panel.json")
print("Median", panel_view["probability"]["median"], "| range", panel_view["probability"]["min"], "-", panel_view["probability"]["max"])
for t in panel_view["triggers"]:
    print(f'{t["agree"]} of {t["of"]} agree: {t["title"]}')

Saved c:\Users\iliev\Documents\github\HFG8-ACAPS-Ground-Truth\data\synthetic\demo_R1043_panel.json
Median 0.7 | range 0.5 - 0.9
6 of 7 agree: Expansion of cross-border attacks and insecurity
5 of 7 agree: Attacks on strategic infrastructure in Chad
3 of 7 agree: Renewed proxy mobilisation and cross-border armed group activity


Tell Diana where the file is. The group view should use the aggregated fields (`probability`, `triggers`, `gaps`). Per-expert `answers` are for the moderator view only, and should never be shown to peers or the public.